In [1]:
import numpy as np
import pandas as pd

# 1. Define the lookback periods (z)
z = np.array([1, 6, 13, 26])

# 2. Define a range of fear values (f) from 0 to 1
f_values = np.linspace(0, 1, 11)  # 0.0, 0.1, ..., 1.0

results = []

for f in f_values:
    # Calculate the raw exponential weights
    # When z is max(z), the exponent is 0, so e^0 = 1.0
    raw_weights = np.exp(-f * (np.max(z) - z))
    
    # Normalize so the sum of weights is always 1.0
    normalized_weights = raw_weights / np.sum(raw_weights)
    
    results.append({
        'fear_factor': round(f, 2),
        '1_period': normalized_weights[0],
        '6_period': normalized_weights[1],
        '13_period': normalized_weights[2],
        '26_period': normalized_weights[3],
        'sum': np.sum(normalized_weights)
    })

# 3. Display as a clean DataFrame
df_weights = pd.DataFrame(results)
print("Weight Distribution by Fear Factor (f):")
display(df_weights.style.format("{:.4f}", subset=df_weights.columns[1:]))

# Optional: Quick plot to see the transition
df_weights.set_index('fear_factor')[[f'{p}_period' for p in z]].plot(
    title="Weight Shift: Short-term vs Long-term",
    ylabel="Weight Percentage",
    grid=True
)

Weight Distribution by Fear Factor (f):


AttributeError: The '.style' accessor requires jinja2

In [35]:
import numpy as np
import pandas as pd
import plotly.express as px

# 1. Define the parameters
a_sensitivity = 0.1
periods = np.array([1, 2,4,8])
max_z = np.max(periods)

# 2. Setup fear factor range (f) from 0 to 1
f_range = np.linspace(-2.5, 2.5, 101)

plot_data = []

# 3. Generate data: Apply decay sensitivity 'a'
for f in f_range:
    # Formula: raw = e^(-a * f * (max(z) - z))
    raw_weights = np.exp(-a_sensitivity * f * (max_z - periods))
    
    # Normalize
    normalized_weights = raw_weights / np.sum(raw_weights)
    
    for i, p in enumerate(periods):
        plot_data.append({
            'Fear Factor (f)': f,
            'Weight': normalized_weights[i],
            'Window': f'{p} Periods'
        })

df_plot = pd.DataFrame(plot_data)

# 4. Create Plotly Visualization
fig = px.line(
    df_plot, 
    x='Fear Factor (f)', 
    y='Weight', 
    color='Window',
    title=f'Subtle Weight Transition: Sensitivity Factor a = {a_sensitivity}',
    labels={'Weight': 'Portfolio Weight (%)', 'Fear Factor (f)': 'Fear Factor Index'},
    template='plotly_white'
)

# Enhance plot presentation
fig.update_layout(
    hovermode="x unified",
    # Clamp y-axis to focus on the 25% starting point
    yaxis=dict(tickformat=".0%", range=[0, 1]),
    legend_title_text='Lookback Period'
)

# Add annotations to highlight the endpoints
# Fear = 0 (Starts at 25%)
fig.add_annotation(x=0.01, y=0.25, text="f=0: Start (All 25%)", showarrow=True, arrowhead=1)

# Fear = 1 (Endpoint weights calculated at 13%, 17%, 22%, 48%)
fig.add_annotation(x=0.99, y=0.48, text="f=1: Max Shift", showarrow=True, arrowhead=1, ax=-40, ay=-30)

fig.show()

In [22]:
import numpy as np
import pandas as pd
import plotly.express as px

def get_two_param_weights(fear_val, k=10, x0=0.5):
    """
    fear_val: raw fear (0 to 1)
    k: steepness of the shift (higher = faster transition)
    x0: midpoint threshold (where the shift is 50% complete)
    """
    z = np.array([1, 6, 13, 26])
    
    # 1. Transform fear through a sigmoid
    # This creates the "delayed" reaction effect
    transformed_f = 1 / (1 + np.exp(-k * (fear_val - x0)))
    
    # 2. Apply the exponential decay using the transformed fear
    raw_weights = np.exp(-transformed_f * (np.max(z) - z))
    
    # 3. Normalize
    return raw_weights / np.sum(raw_weights)

# --- Simulation ---
fear_range = np.linspace(-2.5, 2.5, 100)
# Example: High threshold (0.7) and sharp transition (k=20)
results = [get_two_param_weights(f, k=0, x0=100000) for f in fear_range]

# Plotting...
df = pd.DataFrame(results, columns=['1p', '6p', '13p', '26p'])
df['Fear'] = fear_range
fig = px.line(df, x='Fear', y=['1p', '6p', '13p', '26p'], 
             title=f"Two-Parameter Shift (Midpoint={0.7}, Steepness={20})")
fig.show()

In [32]:
import numpy as np
import pandas as pd
import plotly.express as px

# 1. Define the Two-Parameter Sigmoid Weight Function
def get_dynamic_weights(fear_val, k, x0_inv):
    """
    fear_val: Range 0 to 1
    k: Steepness (Sensitivity)
    x0: Midpoint (Threshold)
    """
    z = np.array([1, 6, 13, 26])
    max_z = np.max(z)
    
    # The transformation function T(f)
    # This determines the "intensity" of the decay
    tf = 1 / (1 + np.exp(-k * (fear_val - 1/max(x0_inv, .00000001))))
    # Calculate raw weights: e^(-tf * distance_from_26)
    # tf=0 -> e^0 = 1 (Equal weights)
    # tf=1 -> e^-(max_z - z) (Maximum decay)
    raw_weights = np.exp(-tf * (max_z - z))
    
    # Normalize to sum to 1.0
    return raw_weights / np.sum(raw_weights)

# 2. Configuration Parameters (Modify these to see the shift)
K_STEEPNESS = 10 # Higher = sharper transition
X0_MIDPOINT = 100   # 0.5 = shift happens in the middle of the fear range

# 3. Generate Data
fear_range = np.linspace(0, 1, 101)
periods = [1, 6, 13, 26]
data = []

for f in fear_range:
    w = get_dynamic_weights(f, K_STEEPNESS, X0_MIDPOINT)
    for i, period in enumerate(periods):
        data.append({
            'Market Fear': f,
            'Weight': w[i],
            'Window': f'{period} Periods'
        })

df_plot = pd.DataFrame(data)

# 4. Create Plotly Figure
fig = px.line(
    df_plot, 
    x='Market Fear', 
    y='Weight', 
    color='Window',
    title=f'Weight Shift Model (Steepness k={K_STEEPNESS}, Midpoint x0={X0_MIDPOINT})',
    labels={'Weight': 'Portfolio Allocation', 'Market Fear': 'Normalized Fear Factor'},
    template='plotly_white'
)

fig.update_layout(
    hovermode="x unified",
    yaxis=dict(tickformat=".0%", range=[0, 1]),
    legend_title_text='Lookback Window'
)

fig.show()